# Road injuries: exploration

Data: IHME GBD 2023, road injury deaths, 204 countries (see `data/SOURCES.md`).

Questions to answer:
1. Where are roads deadliest?
2. Where is the biggest number of deaths by road accidents?
3. Who dies on the roads? Pedestrians, cyclists, car occupants and how do they differ per country?
4. How big is the female - male gap?
5. Where do road accidents land compared to other external causes?
6. Do safety laws go with fewer deaths?
7. Are road accidents in the Republic of Moldova just generally more published on news websites compared to other countries with similar profile?

## Setup

In [ ]:
import plotly.graph_objects as go
import polars as pl

from howlikely.load import load_road_data

In [ ]:
df = load_road_data()
road = df.filter(cause="Road injuries")  # parent cause; sub-types (pedestrian, ...) come later

df.head()

## Q1. Where are roads deadliest?

Age-standardized death rate per 100,000, both sexes. The line shows the uncertainty interval (lower to upper bound).

In [ ]:
rate_both = (
    road.filter(age="Age-standardized", sex="Both")
    .select("country", "rate", "rate_lower", "rate_upper")
    .sort("rate")  # ascending -> highest ends up at the top in plotly
)

# Connector lines: x pairs separated by None so plotly breaks the line between rows
line_x, line_y, line_label = [], [], []
for c, low, up in rate_both.select("country", "rate_lower", "rate_upper").iter_rows():
    line_x += [low, up, None]
    line_y += [c, c, None]
    line_label += ["Lower bound", "Upper bound", None]

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=line_x,
        y=line_y,
        mode="lines+markers",
        line=dict(color="#c3c2b7", width=2),
        marker=dict(color="#e76430", size=8, line=dict(color="white", width=2)),
        text=line_label,
        hovertemplate="%{y}<br>%{text}: %{x:.1f} per 100k<extra></extra>",
        showlegend=False,
    )
)
fig.add_trace(
    go.Scatter(
        x=rate_both["rate"].to_list(),
        y=rate_both["country"].to_list(),
        mode="markers",
        marker=dict(color="#2a78d6", size=8, line=dict(color="white", width=2)),
        hovertemplate="%{y}<br>Rate: %{x:.1f} per 100k<extra></extra>",
    )
)
fig.update_layout(
    title="Road injury deaths per 100,000 with lower and upper bound, age-standardized, 2023 (GBD)",
    height=len(rate_both) * 16 + 150,  # ~16px per country
    xaxis=dict(title="Deaths per 100,000", side="top", gridcolor="#eeeeee"),
    yaxis=dict(tickfont=dict(size=10), gridcolor="#f5f5f5"),
    plot_bgcolor="white",
    margin=dict(l=180),
)
fig.show()

**Findings:** Countries with higher rates have also an increased width of the lower to upper bounds.

### How certain are these rates?

Relative width = (upper - lower) / rate. A high value means GBD is unsure about the estimate, usually because the country registers few deaths and the rate is modeled.

In [ ]:
uncertainty = (
    road.filter(age="Age-standardized", sex="Both")
    .with_columns(rel_width=(pl.col("rate_upper") - pl.col("rate_lower")) / pl.col("rate"))
    .sort("rel_width")  # ascending -> most uncertain at the top in plotly
)
uncertainty.shape

In [ ]:
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=uncertainty["rel_width"].to_list(),
        y=uncertainty["country"].to_list(),
        mode="markers",
        marker=dict(color="#2a78d6", size=8),
    )
)
fig.update_layout(
    title="Countries by relative width of the bounds compared to the rate",
    height=len(uncertainty) * 16 + 150,
    xaxis=dict(title="relative_width", side="top", type="log", tickformat=","),
    yaxis=dict(tickfont=dict(size=10)),
    plot_bgcolor="white",
    margin=dict(l=180),
)
fig.show()

**Findings:** 

## Q2. Where do the most people die?

Number of deaths, all ages, both sexes. Counts show scale, not risk: big countries come out on top.

In [ ]:
TOP_N = 15

deaths = road.filter(sex="Both", age="All ages").sort("deaths", descending=True)
total = deaths["deaths"].sum()

top = deaths.head(TOP_N).select("country", "deaths")
rest = pl.DataFrame(
    {
        "country": [f"Rest of world ({deaths.height - TOP_N} countries)"],
        "deaths": [total - top["deaths"].sum()],
    }
)
bars = pl.concat([top, rest]).reverse()  # reverse -> largest at the top in plotly

fig = go.Figure(
    go.Bar(
        x=bars["deaths"].to_list(),
        y=bars["country"].to_list(),
        orientation="h",
        marker_color=["#c3c2b7"] + ["#2a78d6"] * TOP_N,  # rest of world in gray
        hovertemplate="%{y}: %{x:,.0f} deaths<extra></extra>",
    )
)
fig.update_layout(
    title="Road injury deaths, 2023: 10 countries account for 58% of the world's total",
    xaxis=dict(title="Deaths", tickformat=",", gridcolor="#eeeeee"),
    plot_bgcolor="white",
    height=550,
    margin=dict(l=220),
)
fig.show()

In [ ]:
deaths_table = deaths.select(
    "country",
    pl.col("deaths").round(0),
    share=(pl.col("deaths") / total * 100).round(2),
    cumulative=(pl.col("deaths").cum_sum() / total * 100).round(1),
)

with pl.Config(tbl_rows=-1):  # -1 = print every row
    display(deaths_table)

**Findings:** 

## Q4. How big is the female - male gap?

Age-standardized rate per 100,000 for each sex, plus both sexes combined.

In [ ]:
# One row per country: female, male and both-sexes rate side by side
rate_by_sex = (
    road.filter(age="Age-standardized")
    .pivot(on="sex", index="country", values="rate")
    .sort("Both")  # ascending -> highest ends up at the top in plotly
)

# Connector lines: x pairs separated by None so plotly breaks the line between rows
line_x, line_y = [], []
for c, f, m in rate_by_sex.select("country", "Female", "Male").iter_rows():
    line_x += [f, m, None]
    line_y += [c, c, None]

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=line_x,
        y=line_y,
        mode="lines",
        line=dict(color="#c3c2b7", width=2),
        hoverinfo="skip",
        showlegend=False,
    )
)
for sex, color in [("Female", "#eb6834"), ("Male", "#2a78d6"), ("Both", "#898781")]:
    fig.add_trace(
        go.Scatter(
            x=rate_by_sex[sex].to_list(),
            y=rate_by_sex["country"].to_list(),
            mode="markers",
            name=sex,
            marker=dict(color=color, size=8, line=dict(color="white", width=2)),
            hovertemplate="%{y}<br>" + sex + ": %{x:.1f} per 100k<extra></extra>",
        )
    )
fig.update_layout(
    title="Road injury deaths per 100,000, age-standardized, 2023 (GBD)",
    height=len(rate_by_sex) * 16 + 150,  # ~16px per country
    xaxis=dict(title="Deaths per 100,000", side="top", gridcolor="#eeeeee"),
    yaxis=dict(tickfont=dict(size=10), gridcolor="#f5f5f5"),
    legend=dict(orientation="h", y=1.02, x=0),
    plot_bgcolor="white",
    margin=dict(l=180),
)
fig.show()

In [ ]:
for c, m, f in rate_by_sex.select("country", "Male", "Female").iter_rows():
    if f >= m:
        print(f"{c}<br>Male Rate: {m} | {f} :Female Rate")

**Findings:** In absolutely each country the rate for Males is higher than for Females